In [191]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, chi2, mutual_info_classif, SelectKBest

In [192]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [193]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [194]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Dropping `Cabin` feature as it has large number of missing values

In [195]:
df.drop('Cabin',axis = 1,inplace = True)

Imputing `Age` and `Embarked` Features using SimpleImputer

In [196]:
impute = SimpleImputer(strategy = 'median')
df['Age'] = impute.fit_transform(df[['Age']])

In [197]:
imputer = SimpleImputer(strategy = 'most_frequent')
df['Embarked'] = imputer.fit_transform(df[['Embarked']]).ravel()

In [198]:
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

- Separating Numerical and Categorical Columns using `select_dtypes`
- setting `Survived` Feature as Target

In [199]:
numerical_cols = df.select_dtypes(exclude = 'object').columns
categorical_cols = df.select_dtypes(include = ['object','str']).columns
y = df['Survived']
print(numerical_cols)
print(categorical_cols)

Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='str')
Index(['Name', 'Sex', 'Ticket', 'Embarked'], dtype='str')


Creating Separate Data Frames of Numerical and Categorical
- Dropping `Name` and `Ticket` Features as they add irrelevant features when encoded
- Encoding Categorical Data
- Dropping `Target` from the Features

In [200]:
cat_df = df[categorical_cols].copy()
num_df = df[numerical_cols].copy()
cat_df.drop(['Name','Ticket'],axis = 1,inplace = True)
num_df.drop('Survived',axis = 1,inplace = True)
cat_encoded = pd.get_dummies(cat_df)
cat_encoded.head()

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,False,True,False,False,True
1,True,False,True,False,False
2,True,False,False,False,True
3,True,False,False,False,True
4,False,True,False,False,True


Applying `Chi2` on Categorical Data Frame After Encoding

In [201]:
chi_selector = SelectKBest(chi2,k = 1)
mi_selector = SelectKBest(mutual_info_classif,k=1)
cat_selected = chi_selector.fit_transform(cat_encoded,y)
cat_selected_df = pd.DataFrame(cat_selected,columns=cat_encoded.columns[chi_selector.get_support()])
cat_selected_df.head()

,Sex_female
0,False
1,True
2,True
3,True
4,False


Applying `mutual_info_classif` on Numerical Data Frame

In [202]:
num_selected = mi_selector.fit_transform(num_df,y)
num_selected_df = pd.DataFrame(num_selected,columns = num_df.columns[mi_selector.get_support()])
num_selected_df.head()

,Fare
0,7.2500
1,71.2833
2,7.9250
3,53.1000
4,8.0500


Creating Separate Data Frames to Interpret the Scores Assigned 

In [203]:
cat_score_df = pd.DataFrame({
    'Features' : cat_encoded.columns,
    'scores' : chi_selector.scores_
})
cat_score_df.head()

,Features,scores
0,Sex_female,170.348127
1,Sex_male,92.702447
2,Embarked_C,20.464401
3,Embarked_Q,0.010847
4,Embarked_S,5.489205


In [204]:
num_score_df = pd.DataFrame({
    'Features' : num_df.columns,
    'Scores' : mi_selector.scores_
})
num_score_df

,Features,Scores
0,PassengerId,0.014367
1,Pclass,0.052979
2,Age,0.056067
3,SibSp,0.004099
4,Parch,0.004667
5,Fare,0.136534
